# Mouse Fig 2c pre-peak bump

M_002 only (blocks 012–015). Current 2c is **unsigned angular speed**, **peak-aligned**,
window **±100 ms**, 2 ms grid, Gaussian `bandwidth_ms=10`. Every finite sample in
`[t0−100, t0+100]` is accumulated — **other saccades are not masked**.

**Question:** is the ~−80 ms speed bump a leftover previous saccade in that window,
or a biological ~12.5 Hz rhythm?

**This notebook does not change** `export_pos_vel_bundle`. It writes a candidate
cleaned PDF plus diagnostics.

Kernel: `eye_repo_mac`. From the repo root, `PYTHONPATH=src`. Lab volume must be mounted.


## 0. Setup


In [3]:
%matplotlib inline
from __future__ import annotations

import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt

plt.ioff()

REPO = Path.cwd()
if not (REPO / "src" / "eye_tracking_system_tools").is_dir():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "eye_tracking_system_tools").is_dir():
            REPO = p
            break

sys.path.insert(0, str(REPO / "src"))
os.environ.setdefault("MPLCONFIGDIR", str(REPO / ".mplconfig"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(exist_ok=True)

from eye_tracking_system_tools.analysis.block_registry import load_registry
from eye_tracking_system_tools.analysis.event_cache import build_or_load_event_tables
from eye_tracking_system_tools.analysis.export_meta import load_params_yaml
from eye_tracking_system_tools.analysis.review_collect import review_answers_dir
from eye_tracking_system_tools.analysis.run_layout import resolve_figure_dirs

# Empty TAG overwrites outputs/review_answers_latest
TAG = ""
REGISTRY = REPO / "configs" / "mouse_M_002_blocks.yaml"
PARAMS_YAML = REPO / "configs" / "analysis_params_mouse.yaml"
run = review_answers_dir(REPO / "outputs", TAG)
OUT = run / "mouse_2c_prepeak"
figures_dir, metadata_dir = resolve_figure_dirs(OUT)
# Event cache lives on the run (shared), not inside the plot folder.
cache_dir = run / "metadata"
cache_dir.mkdir(parents=True, exist_ok=True)
print("REPO :", REPO)
print("run  :", run)
print("plot :", OUT)
print("  figures :", figures_dir)
print("  metadata:", metadata_dir)


REPO : /Users/nimi/Projects/PETS
run  : /Users/nimi/Projects/PETS/outputs/review_answers_latest
plot : /Users/nimi/Projects/PETS/outputs/review_answers_latest/mouse_2c_prepeak
  figures : /Users/nimi/Projects/PETS/outputs/review_answers_latest/mouse_2c_prepeak/plots
  metadata: /Users/nimi/Projects/PETS/outputs/review_answers_latest/mouse_2c_prepeak/metadata


## 1. PARAMS (fidget surface)

Re-run from here down after edits. Defaults match current mouse 2c.
`ISOLATION_MODE` chooses the **candidate** `figure_2c_isolated.pdf` (diagnostics always
compute `none`, `drop_events`, and `nan_mask_neighbors`).


In [5]:
# --- window / kernel (mouse 2c defaults) ---
T_WINDOW_MS = (-100.0, 100.0)   # averaging window around alignment time
DT_MS = 2.0
BANDWIDTH_MS = 10.0             # also try 0 (off) and 4 (review README)
ALIGN_TO = "peak"               # "peak" | "onset"
SMOOTHING = True
SUBSAMPLE_PEAK = False

# --- amplitude bins ---
AMP_COL = "net_angular_disp"
BIN_WIDTH_DEG = 5.0
MIN_AMP_DEG = 0.5
MAX_AMP_PCT = 99.5
MIN_EVENTS_PER_BIN = 15
PER_ANIMAL_MAX_BINS = 8
VELOCITY_UNIT = "deg/sec"

# --- isolation (candidate PDF uses ISOLATION_MODE) ---
# none                : current 2c (no neighbor handling)
# drop_events         : keep only saccades with no other same-eye peak in ±NEIGHBOR_RADIUS_MS
# nan_mask_neighbors  : keep all events; NaN samples that fall inside another event's
#                       [on − NEIGHBOR_PAD_MS, off + NEIGHBOR_PAD_MS], then occupancy-average
ISOLATION_MODE = "nan_mask_neighbors"
NEIGHBOR_RADIUS_MS = 100.0
NEIGHBOR_PAD_MS = 0.0
MIN_NEIGHBOR_AMP_DEG = 0.5

# --- bump search on the mean ---
PRE_SEARCH_MS = (-100.0, -40.0)
POST_SEARCH_MS = (40.0, 100.0)

# --- detection (slow; leave False unless isolation fails) ---
FORCE_REDETECT = False
SPEED_THRESHOLD_DEG_PER_FRAME = 0.8

# --- output ---
SHOW_INLINE = True
N_EXAMPLE_TRACES = 6


## 2. Load event tables (keep traces)


In [7]:
from copy import deepcopy

import numpy as np
import pandas as pd

params = load_params_yaml(PARAMS_YAML)
if FORCE_REDETECT:
    sacc = dict(params.get("saccade") or {})
    sacc["speed_threshold_deg_per_frame"] = float(SPEED_THRESHOLD_DEG_PER_FRAME)
    params = {**params, "saccade": sacc}

ms = dict(params.get("main_sequence") or {})
# Overlay fidget knobs using the exporter's key names (mouse YAML may use aliases).
ms.update(
    {
        "amp_col": AMP_COL,
        "bin_width_deg": BIN_WIDTH_DEG,
        "min_amp_deg": MIN_AMP_DEG,
        "max_amp_pct": MAX_AMP_PCT,
        "t_window_ms": list(T_WINDOW_MS),
        "dt_ms": DT_MS,
        "bandwidth_ms": BANDWIDTH_MS,
        "min_events_per_bin_traces": MIN_EVENTS_PER_BIN,
        "per_animal_max_bins": PER_ANIMAL_MAX_BINS,
        "smoothing": SMOOTHING,
        "subsample_peak": SUBSAMPLE_PEAK,
        "align_to": ALIGN_TO,
        "velocity_unit": VELOCITY_UNIT,
        "plot_animals": None,
    }
)
params = {**params, "main_sequence": ms}

specs = load_registry(REGISTRY)
tables, cache_path, from_cache = build_or_load_event_tables(
    specs,
    params,
    cache_dir,
    keep_traces=True,
    force=FORCE_REDETECT,
    prefer_finalized=not FORCE_REDETECT,
)
print(
    f"blocks={len(tables.blocks)}  events={len(tables.all_saccades)}  "
    f"cache={'hit' if from_cache else 'miss'}"
)
print(cache_path)
print("animals:", sorted(tables.all_saccades["animal"].astype(str).unique().tolist()) if not tables.all_saccades.empty else [])
print("columns:", sorted(tables.all_saccades.columns.tolist())[:40] if not tables.all_saccades.empty else [])


[M_002_block_012] eye CSVs: L=left_eye_data_raw_verified.csv (raw_verified), R=right_eye_data_raw_verified.csv (raw_verified)
[M_002_block_013] eye CSVs: L=left_eye_data_raw_verified.csv (raw_verified), R=right_eye_data_raw_verified.csv (raw_verified)
[M_002_block_014] eye CSVs: L=left_eye_data_raw_verified.csv (raw_verified), R=right_eye_data_raw_verified.csv (raw_verified)
[M_002_block_015] eye CSVs: L=left_eye_data_raw_verified.csv (raw_verified), R=right_eye_data_raw_verified.csv (raw_verified)
blocks=4  events=17585  cache=hit
/Users/nimi/Projects/PETS/outputs/review_answers_latest/metadata/event_cache/c063f64cd1604c07c521dc977d99a9a697e4f86a.pkl
animals: ['M_002']
columns: ['Main', 'Sub', 'animal', 'bad_detections', 'block', 'concurrency', 'delta_phi', 'delta_theta', 'diameter_profile', 'eye', 'head_movement', 'length', 'magnitude_pixel', 'magnitude_raw_angular', 'magnitude_raw_pixel', 'net_angular_disp', 'overall_angle_deg', 'peak_velocity', 'phi_end_pos', 'phi_init_pos', 'sacca

## 3. Helpers (local 2c kernel + isolation)


In [9]:
from collections import defaultdict

from scipy.ndimage import gaussian_filter1d

from eye_tracking_system_tools.analysis.figure_display import show_and_close
from eye_tracking_system_tools.analysis.figures_2c_2e import (
    _bin_viridis_colors,
    _ensure_eye_traces,
    _eye_dataframe,
    _peak_time_in_window,
    _plot_pos_vel_pdfs,
    _raw_velocity_trace,
)

tables = _ensure_eye_traces(tables, tables.all_saccades)

tmin, tmax = float(T_WINDOW_MS[0]), float(T_WINDOW_MS[1])
t_grid = np.arange(tmin, tmax + DT_MS * 0.5, DT_MS, dtype=np.float64)
nb = len(t_grid)
sigma_bins = max(1.0, float(BANDWIDTH_MS) / float(DT_MS)) if (SMOOTHING and BANDWIDTH_MS > 0) else 0.0


def _save_fig(fig, path: Path, *, show: bool = SHOW_INLINE) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, format="pdf", bbox_inches="tight")
    show_and_close(fig, show=show)
    return path


def _group_key(animal, block, eye) -> tuple[str, str, str]:
    return str(animal), str(block), str(eye).upper()[:1]


def build_eye_cache(tables, events: pd.DataFrame) -> dict:
    cache: dict[tuple[str, str, str], dict | None] = {}
    keys = {
        _group_key(a, b, e)
        for a, b, e in zip(events["animal"], events["block"], events["eye"])
    }
    for animal, block, eye in sorted(keys):
        df = _eye_dataframe(tables, animal, block, eye)
        if df is None or not {"ms_axis", "k_phi", "k_theta"}.issubset(df.columns):
            cache[(animal, block, eye)] = None
            continue
        t = df["ms_axis"].to_numpy(dtype=float)
        phi = df["k_phi"].to_numpy(dtype=float)
        th = df["k_theta"].to_numpy(dtype=float)
        t_ms, v_vel = _raw_velocity_trace(df, velocity_unit=VELOCITY_UNIT)
        dt_s = np.diff(t) / 1000.0
        v_phi = np.divide(np.diff(phi), dt_s, out=np.full_like(dt_s, np.nan), where=dt_s > 0)
        v_th = np.divide(np.diff(th), dt_s, out=np.full_like(dt_s, np.nan), where=dt_s > 0)
        cache[(animal, block, eye)] = {
            "t": t,
            "phi": phi,
            "th": th,
            "t_ms": t_ms,
            "v_vel": v_vel,
            "v_phi": v_phi,
            "v_th": v_th,
        }
    return cache


def attach_peaks(events: pd.DataFrame, eye_cache: dict) -> pd.DataFrame:
    ev = events.copy()
    peaks = np.full(len(ev), np.nan, dtype=float)
    for i, row in enumerate(ev.itertuples(index=False)):
        packed = eye_cache.get(_group_key(row.animal, row.block, row.eye))
        if packed is None:
            continue
        on_t = float(row.saccade_on_ms)
        off_t = float(row.saccade_off_ms)
        pk = _peak_time_in_window(
            packed["t_ms"], packed["v_vel"], on_t, off_t, subsample_peak=SUBSAMPLE_PEAK
        )
        peaks[i] = pk if np.isfinite(pk) else on_t
    ev["peak_ms"] = peaks
    return ev


def annotate_neighbors(events: pd.DataFrame, *, radius_ms: float, min_amp: float) -> pd.DataFrame:
    ev = events.copy()
    n = len(ev)
    prev_isi = np.full(n, np.nan)
    next_isi = np.full(n, np.nan)
    prev_peak_lag = np.full(n, np.nan)
    next_peak_lag = np.full(n, np.nan)
    prev_amp = np.full(n, np.nan)
    next_amp = np.full(n, np.nan)
    roles = np.array(["isolated"] * n, dtype=object)
    amp = pd.to_numeric(ev[AMP_COL], errors="coerce").to_numpy(dtype=float)
    big = np.isfinite(amp) & (amp >= float(min_amp))

    # Stable 0..n-1 positions for numpy assignment (ev may carry a non-range index).
    ev = ev.reset_index(drop=True)
    for _, g in ev.groupby(["animal", "block", "eye"], sort=False):
        pos = g.index.to_numpy()
        if pos.size == 0:
            continue
        on = g["saccade_on_ms"].to_numpy(dtype=float)
        pk = g["peak_ms"].to_numpy(dtype=float)
        a = amp[pos]
        use = big[pos] & np.isfinite(on) & np.isfinite(pk)
        if not np.any(use):
            continue
        order = np.argsort(on)
        on_s, pk_s, a_s, pos_s, use_s = on[order], pk[order], a[order], pos[order], use[order]
        valid = np.where(use_s)[0]
        for k, vi in enumerate(valid):
            i_pos = int(pos_s[vi])
            if k > 0:
                pj = valid[k - 1]
                prev_isi[i_pos] = on_s[vi] - on_s[pj]
                prev_peak_lag[i_pos] = pk_s[vi] - pk_s[pj]
                prev_amp[i_pos] = a_s[pj]
            if k + 1 < len(valid):
                nj = valid[k + 1]
                next_isi[i_pos] = on_s[nj] - on_s[vi]
                next_peak_lag[i_pos] = pk_s[nj] - pk_s[vi]
                next_amp[i_pos] = a_s[nj]
            has_p = np.isfinite(prev_peak_lag[i_pos]) and prev_peak_lag[i_pos] <= radius_ms
            has_n = np.isfinite(next_peak_lag[i_pos]) and next_peak_lag[i_pos] <= radius_ms
            if has_p and has_n:
                roles[i_pos] = "middle"
            elif has_p:
                roles[i_pos] = "last"
            elif has_n:
                roles[i_pos] = "first"
            else:
                roles[i_pos] = "isolated"

    ev["prev_isi_ms"] = prev_isi
    ev["next_isi_ms"] = next_isi
    ev["prev_peak_lag_ms"] = prev_peak_lag
    ev["next_peak_lag_ms"] = next_peak_lag
    ev["prev_amp"] = prev_amp
    ev["next_amp"] = next_amp
    ev["burst_role"] = roles
    ev["has_prev"] = np.isfinite(prev_peak_lag) & (prev_peak_lag <= radius_ms)
    ev["has_next"] = np.isfinite(next_peak_lag) & (next_peak_lag <= radius_ms)
    return ev


def _neighbor_intervals(group: pd.DataFrame, *, pad_ms: float, min_amp: float) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    amp = pd.to_numeric(group[AMP_COL], errors="coerce").to_numpy(dtype=float)
    keep = np.isfinite(amp) & (amp >= float(min_amp))
    on = group["saccade_on_ms"].to_numpy(dtype=float)[keep] - float(pad_ms)
    off = group["saccade_off_ms"].to_numpy(dtype=float)[keep] + float(pad_ms)
    pk = group["peak_ms"].to_numpy(dtype=float)[keep]
    return on, off, pk


def _mask_other_events(t_abs: np.ndarray, on_i: float, off_i: float, ons: np.ndarray, offs: np.ndarray) -> np.ndarray:
    # True where sample is NOT inside another event's padded [on, off].
    # Vectorized over neighbors; t_abs should already be the ±window samples.
    if ons.size == 0 or t_abs.size == 0:
        return np.ones(t_abs.shape, dtype=bool)
    self_m = (np.abs(ons - on_i) < 1e-6) & (np.abs(offs - off_i) < 1e-6)
    ons_o = ons[~self_m]
    offs_o = offs[~self_m]
    if ons_o.size == 0:
        return np.ones(t_abs.shape, dtype=bool)
    hit = (t_abs[:, None] >= ons_o[None, :]) & (t_abs[:, None] <= offs_o[None, :])
    return ~hit.any(axis=1)


def _smooth_div(num: np.ndarray, den: np.ndarray) -> np.ndarray:
    if sigma_bins > 0:
        num_s = gaussian_filter1d(num, sigma=sigma_bins, mode="nearest")
        den_s = gaussian_filter1d(den, sigma=sigma_bins, mode="nearest")
        out = np.divide(num_s, den_s, out=np.full(nb, np.nan), where=den_s > 1e-6)
    else:
        out = np.divide(num, den, out=np.full(nb, np.nan), where=den > 1e-6)
    return np.clip(out, 0.0, None)


def _amp_edges(amp: np.ndarray) -> tuple[np.ndarray, list[str]]:
    amp = amp[np.isfinite(amp) & (amp >= MIN_AMP_DEG)]
    if amp.size == 0:
        return np.array([0.0, BIN_WIDTH_DEG]), ["0-5°"]
    max_amp = float(np.nanpercentile(amp, MAX_AMP_PCT))
    edges = np.arange(0.0, max_amp + BIN_WIDTH_DEG, BIN_WIDTH_DEG)
    if len(edges) > PER_ANIMAL_MAX_BINS + 1:
        edges = edges[: PER_ANIMAL_MAX_BINS + 1]
    labels = [f"{int(edges[i])}-{int(edges[i + 1])}°" for i in range(len(edges) - 1)]
    return edges, labels


def average_2c(
    events: pd.DataFrame,
    eye_cache: dict,
    *,
    isolation: str = "none",
    row_mask: np.ndarray | None = None,
    signed: bool = False,
) -> dict:
    # Occupancy-weighted mean unsigned (or signed-axis) speed by amp bin.
    sub = events.copy()
    if row_mask is not None:
        sub = sub.loc[row_mask].copy()
    amp = pd.to_numeric(sub[AMP_COL], errors="coerce")
    sub = sub.loc[amp >= MIN_AMP_DEG].copy()
    if sub.empty:
        return {"t_grid": t_grid, "series": [], "n_used": 0, "n_dropped": 0}

    edges, labels = _amp_edges(sub[AMP_COL].to_numpy(dtype=float))
    sub["amp_bin"] = pd.cut(sub[AMP_COL], bins=edges, labels=labels, include_lowest=True, right=False)

    # Neighbor intervals per eye (for nan_mask / drop_events).
    interval_map: dict[tuple[str, str, str], tuple[np.ndarray, np.ndarray, np.ndarray]] = {}
    for key, g in events.groupby(["animal", "block", "eye"], sort=False):
        interval_map[_group_key(*key)] = _neighbor_intervals(
            g, pad_ms=NEIGHBOR_PAD_MS, min_amp=MIN_NEIGHBOR_AMP_DEG
        )

    series: list[dict] = []
    n_used_total = 0
    n_dropped_total = 0
    for lab in labels:
        g = sub[sub["amp_bin"] == lab]
        if len(g) < MIN_EVENTS_PER_BIN:
            continue
        v_num = np.zeros(nb)
        v_den = np.zeros(nb)
        s_num = np.zeros(nb)
        s_den = np.zeros(nb)
        n_used = 0
        n_dropped = 0
        for row in g.itertuples(index=False):
            packed = eye_cache.get(_group_key(row.animal, row.block, row.eye))
            if packed is None or packed["t_ms"].size < 3:
                n_dropped += 1
                continue
            t = packed["t"]
            phi = packed["phi"]
            th = packed["th"]
            t_ms = packed["t_ms"]
            v_vel = packed["v_vel"]
            on_t = float(row.saccade_on_ms)
            off_t = float(row.saccade_off_ms)
            on_i = int(np.argmin(np.abs(t - on_t)))
            off_i = int(np.argmin(np.abs(t - off_t)))
            if off_i <= on_i:
                n_dropped += 1
                continue
            dphi = phi[off_i] - phi[on_i]
            dth = th[off_i] - th[on_i]
            amp_i = float(np.hypot(dphi, dth))
            if not np.isfinite(amp_i) or amp_i <= 0:
                n_dropped += 1
                continue
            ux, uy = dphi / amp_i, dth / amp_i
            t0 = float(row.peak_ms) if ALIGN_TO == "peak" else on_t
            if not np.isfinite(t0):
                n_dropped += 1
                continue

            ons, offs, pks = interval_map[_group_key(row.animal, row.block, row.eye)]
            if isolation == "drop_events":
                others = pks[np.isfinite(pks) & (np.abs(pks - t0) > 1e-6)]
                if np.any(np.abs(others - t0) <= NEIGHBOR_RADIUS_MS):
                    n_dropped += 1
                    continue

            m = (t_ms >= t0 + tmin) & (t_ms <= t0 + tmax) & np.isfinite(v_vel)
            if isolation == "nan_mask_neighbors" and np.any(m):
                extra = _mask_other_events(
                    t_ms[m],
                    on_t - float(NEIGHBOR_PAD_MS),
                    off_t + float(NEIGHBOR_PAD_MS),
                    ons,
                    offs,
                )
                m_idx = np.flatnonzero(m)
                m[m_idx] = extra
            if not np.any(m):
                n_dropped += 1
                continue
            t_rel = t_ms[m] - t0
            v = v_vel[m]
            idx = np.floor((t_rel - tmin) / DT_MS).astype(int)
            ok = (idx >= 0) & (idx < nb) & np.isfinite(v)
            if not np.any(ok):
                n_dropped += 1
                continue
            np.add.at(v_num, idx[ok], v[ok])
            np.add.at(v_den, idx[ok], 1.0)
            if signed:
                v_axis = packed["v_phi"] * ux + packed["v_th"] * uy
                vs = v_axis[m]
                ok_s = ok & np.isfinite(vs)
                if np.any(ok_s):
                    np.add.at(s_num, idx[ok_s], vs[ok_s])
                    np.add.at(s_den, idx[ok_s], 1.0)
            n_used += 1
        if n_used == 0:
            continue
        vel = _smooth_div(v_num, v_den)
        rec = {
            "raw_label": lab,
            "label": f"{lab} (n={n_used})",
            "n_events": int(n_used),
            "n_dropped": int(n_dropped),
            "color_rgba": (0.0, 0.0, 0.0, 1.0),
            "vel_center": vel.astype(np.float32),
            "pos_center": np.full(nb, np.nan, dtype=np.float32),
            "time_axis": t_grid.astype(np.float32),
            "occupancy": v_den.astype(np.float32),
        }
        if signed:
            if sigma_bins > 0:
                sn = gaussian_filter1d(s_num, sigma=sigma_bins, mode="nearest")
                sd = gaussian_filter1d(s_den, sigma=sigma_bins, mode="nearest")
                signed_c = np.divide(sn, sd, out=np.full(nb, np.nan), where=sd > 1e-6)
            else:
                signed_c = np.divide(s_num, s_den, out=np.full(nb, np.nan), where=s_den > 1e-6)
            rec["signed_center"] = signed_c.astype(np.float32)
        series.append(rec)
        n_used_total += n_used
        n_dropped_total += n_dropped
    colors = _bin_viridis_colors(len(series))
    for s, c in zip(series, colors):
        s["color_rgba"] = c
    return {
        "t_grid": t_grid,
        "series": series,
        "n_used": n_used_total,
        "n_dropped": n_dropped_total,
        "isolation": isolation,
    }


def bump_metrics(series_list: list[dict]) -> pd.DataFrame:
    rows = []
    for s in series_list:
        t = np.asarray(s.get("time_axis", t_grid), dtype=float)
        y = np.asarray(s["vel_center"], dtype=float)

        def _peak(lo, hi):
            m = (t >= lo) & (t <= hi) & np.isfinite(y)
            if not np.any(m):
                return np.nan, np.nan
            i = int(np.nanargmax(y[m]))
            return float(t[m][i]), float(y[m][i])

        t_pre, y_pre = _peak(*PRE_SEARCH_MS)
        t_post, y_post = _peak(*POST_SEARCH_MS)
        t_pk, y_pk = _peak(-20.0, 20.0)
        rows.append(
            {
                "bin": s.get("raw_label"),
                "n_events": s.get("n_events"),
                "n_dropped": s.get("n_dropped", 0),
                "t_pre_ms": t_pre,
                "y_pre": y_pre,
                "t_post_ms": t_post,
                "y_post": y_post,
                "y_peak": y_pk,
                "pre_ratio": (y_pre / y_pk) if (np.isfinite(y_pre) and np.isfinite(y_pk) and y_pk > 0) else np.nan,
                "post_ratio": (y_post / y_pk) if (np.isfinite(y_post) and np.isfinite(y_pk) and y_pk > 0) else np.nan,
            }
        )
    return pd.DataFrame(rows)


def plot_speed_panel(series_list, title: str, path: Path, *, ylabel=None, signed=False):
    fig, ax = plt.subplots(figsize=(1.8, 1.8), dpi=300)
    key = "signed_center" if signed else "vel_center"
    for s in series_list:
        y = s.get(key)
        if y is None:
            continue
        ax.plot(s.get("time_axis", t_grid), y, color=s["color_rgba"], lw=1.0, label=s["label"])
    ax.set_title(title, fontsize=9)
    ax.set_xlabel(f"Time from {ALIGN_TO} (ms)", fontsize=8)
    ax.set_ylabel(ylabel or f"Angular speed ({VELOCITY_UNIT})", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_xlim(tmin, tmax)
    if not signed:
        ax.set_ylim(bottom=0)
    fig.tight_layout()
    return _save_fig(fig, path)


def plot_speed_inline(series_list, title: str, path: Path, *, signed=False):
    fig, ax = plt.subplots(figsize=(5.2, 3.4), dpi=120)
    key = "signed_center" if signed else "vel_center"
    for s in series_list:
        y = s.get(key)
        if y is None:
            continue
        ax.plot(s.get("time_axis", t_grid), y, color=s["color_rgba"], lw=1.8, label=s["label"])
    ax.axvline(0.0, color="0.6", lw=0.6, ls="--")
    ax.axvline(-80.0, color="0.75", lw=0.6, ls=":")
    ax.set_title(title)
    ax.set_xlabel(f"Time from {ALIGN_TO} (ms)")
    ax.set_ylabel("Signed axis vel (deg/sec)" if signed else f"Angular speed ({VELOCITY_UNIT})")
    ax.legend(fontsize=7, frameon=False, ncol=2)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_xlim(tmin, tmax)
    fig.tight_layout()
    return _save_fig(fig, path)


## 4. Annotate neighbors and reproduce current 2c


In [11]:
events = tables.all_saccades.copy()
if events.empty:
    raise RuntimeError("all_saccades is empty — check registry paths and detection.")
if AMP_COL not in events.columns:
    raise RuntimeError(f"Missing {AMP_COL!r} in all_saccades: {list(events.columns)}")

eye_cache = build_eye_cache(tables, events)
n_ok = sum(v is not None for v in eye_cache.values())
print(f"eye traces cached: {n_ok}/{len(eye_cache)}")

events = attach_peaks(events, eye_cache)
events = annotate_neighbors(events, radius_ms=NEIGHBOR_RADIUS_MS, min_amp=MIN_NEIGHBOR_AMP_DEG)
events.to_csv(metadata_dir / "event_neighbors.csv", index=False)
print(events[["has_prev", "has_next", "burst_role"]].value_counts(dropna=False).head(20))

current = average_2c(events, eye_cache, isolation="none")
print(f"current 2c: used={current['n_used']} dropped={current['n_dropped']} bins={len(current['series'])}")

animal = str(events["animal"].iloc[0])
bundle = {
    "t_grid": t_grid.astype(np.float32),
    "params": {
        "align_to": ALIGN_TO,
        "velocity_unit": VELOCITY_UNIT,
        "plot_animals": [animal],
        "fig_size": (1.8, 1.8),
        "show_fig_size": (5.2, 3.4),
        "lw": 1.0,
        "legend_ncol": 2,
    },
    "animals": {animal: {"animal": animal, "series": current["series"]}},
}
written = _plot_pos_vel_pdfs(bundle, figures_dir, show=SHOW_INLINE)
print("wrote", {k: str(v) for k, v in written.items()})
plot_speed_inline(current["series"], f"{animal} current 2c (no isolation)", figures_dir / "figure_2c_current_inline.pdf")

bump_now = bump_metrics(current["series"])
bump_now.insert(0, "isolation", "none")
print(bump_now.to_string(index=False))


eye traces cached: 8/8
has_prev  has_next  burst_role
False     False     isolated      4659
          True      first         4351
True      False     last          4351
          True      middle        4224
Name: count, dtype: int64
current 2c: used=16903 dropped=0 bins=6
[2c/2d] M_002: 6 amp bins, 16903 events (pooled across blocks)
         0-5° (n=10497)
         5-10° (n=3573)
         10-15° (n=1650)
         15-20° (n=793)
         20-25° (n=292)
         25-30° (n=98)
<IPython.core.display.Image object>
<IPython.core.display.Image object>
wrote {'figure_2c_M_002.pdf': '/Users/nimi/Projects/PETS/outputs/review_answers_latest/mouse_2c_prepeak/plots/figure_2c_M_002.pdf', 'figure_2c.pdf': '/Users/nimi/Projects/PETS/outputs/review_answers_latest/mouse_2c_prepeak/plots/figure_2c.pdf', 'figure_2c_M_002_legend.pdf': '/Users/nimi/Projects/PETS/outputs/review_answers_latest/mouse_2c_prepeak/plots/figure_2c_M_002_legend.pdf', 'figure_2c_legend.pdf': '/Users/nimi/Projects/PETS/outputs/re

## 5. Neighbor / ISI census


In [13]:
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.0), dpi=120, sharey=True)
for ax, col, title in (
    (axes[0], "prev_peak_lag_ms", "Previous-peak lag"),
    (axes[1], "next_peak_lag_ms", "Next-peak lag"),
):
    x = events[col].to_numpy(dtype=float)
    x = x[np.isfinite(x) & (x > 0) & (x <= 400)]
    ax.hist(x, bins=np.arange(0, 405, 5), color="0.35", histtype="stepfilled", alpha=0.85)
    ax.axvline(80.0, color="C3", lw=1.2, label="80 ms")
    ax.axvline(NEIGHBOR_RADIUS_MS, color="C0", lw=1.0, ls="--", label=f"window {NEIGHBOR_RADIUS_MS:.0f} ms")
    ax.set_title(title)
    ax.set_xlabel("ms")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
axes[0].set_ylabel("Events")
axes[1].legend(fontsize=8, frameon=False)
fig.suptitle("Same-eye peak-to-peak lags (M_002)", fontsize=11)
fig.tight_layout()
_save_fig(fig, figures_dir / "isi_peak_lags.pdf")

# Per amp-bin neighbor fractions
edges, labels = _amp_edges(events[AMP_COL].to_numpy(dtype=float))
ev_b = events.copy()
ev_b["amp_bin"] = pd.cut(ev_b[AMP_COL], bins=edges, labels=labels, include_lowest=True, right=False)
frac_rows = []
for lab in labels:
    g = ev_b[ev_b["amp_bin"] == lab]
    n = max(len(g), 1)
    frac_rows.append(
        {
            "bin": lab,
            "n": int(len(g)),
            "frac_has_prev": float(g["has_prev"].mean()) if len(g) else np.nan,
            "frac_has_next": float(g["has_next"].mean()) if len(g) else np.nan,
            "frac_isolated": float((g["burst_role"] == "isolated").mean()) if len(g) else np.nan,
        }
    )
frac_df = pd.DataFrame(frac_rows)
frac_df.to_csv(metadata_dir / "neighbor_fractions_by_amp_bin.csv", index=False)
print(frac_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(6.0, 3.2), dpi=120)
x = np.arange(len(frac_df))
ax.bar(x - 0.2, frac_df["frac_has_prev"], 0.4, label="has previous in ±radius", color="C1")
ax.bar(x + 0.2, frac_df["frac_has_next"], 0.4, label="has next in ±radius", color="C2")
ax.set_xticks(x)
ax.set_xticklabels(frac_df["bin"], rotation=30, ha="right")
ax.set_ylabel("Fraction of events")
ax.set_ylim(0, 1)
ax.legend(frameon=False, fontsize=8)
ax.set_title("Neighbor occupancy by amplitude bin")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
_save_fig(fig, figures_dir / "neighbor_fractions_by_amp.pdf")

print("burst_role counts:")
print(events["burst_role"].value_counts())
# Onset ISI (classic) in the 2c window
onset = events["prev_isi_ms"].to_numpy(dtype=float)
onset = onset[np.isfinite(onset) & (onset > 0) & (onset <= 200)]
print(f"prev onset ISI in (0, 200] ms: n={onset.size}  median={np.median(onset) if onset.size else np.nan:.1f}  "
      f"frac 60–100 ms={(np.mean((onset >= 60) & (onset <= 100)) if onset.size else np.nan):.3f}")


<IPython.core.display.Image object>
   bin     n  frac_has_prev  frac_has_next  frac_isolated
  0-5° 10497       0.525579       0.479280       0.267981
 5-10°  3573       0.475231       0.513014       0.221942
10-15°  1650       0.452121       0.586061       0.158788
15-20°   793       0.482976       0.605296       0.137453
20-25°   292       0.500000       0.619863       0.113014
25-30°    98       0.551020       0.602041       0.102041
<IPython.core.display.Image object>
burst_role counts:
burst_role
isolated    4659
first       4351
last        4351
middle      4224
Name: count, dtype: int64
prev onset ISI in (0, 200] ms: n=10713  median=50.0  frac 60–100 ms=0.278


## 6. Split averages (decisive test)

Same 2c kernel, **no isolation**. If the −80 ms bump lives only in `has_prev` and
vanishes in `no_prev`, it is contamination from a previous saccade in the window.


In [15]:
split_results = {}
for name, mask in (
    ("no_prev", ~events["has_prev"].to_numpy()),
    ("has_prev", events["has_prev"].to_numpy()),
    ("no_next", ~events["has_next"].to_numpy()),
    ("has_next", events["has_next"].to_numpy()),
    ("isolated", (events["burst_role"] == "isolated").to_numpy()),
    ("first", (events["burst_role"] == "first").to_numpy()),
    ("last", (events["burst_role"] == "last").to_numpy()),
):
    split_results[name] = average_2c(events, eye_cache, isolation="none", row_mask=mask)
    print(f"{name:10s} used={split_results[name]['n_used']:5d}  bins={len(split_results[name]['series'])}")

fig, axes = plt.subplots(2, 2, figsize=(8.4, 6.2), dpi=120, sharex=True, sharey=True)
for ax, name, title in (
    (axes[0, 0], "no_prev", "No previous peak in ±radius"),
    (axes[0, 1], "has_prev", "Has previous peak in ±radius"),
    (axes[1, 0], "no_next", "No next peak in ±radius"),
    (axes[1, 1], "has_next", "Has next peak in ±radius"),
):
    for s in split_results[name]["series"]:
        ax.plot(s["time_axis"], s["vel_center"], color=s["color_rgba"], lw=1.5, label=s["label"])
    ax.axvline(0.0, color="0.6", lw=0.6, ls="--")
    ax.axvline(-80.0, color="0.75", lw=0.6, ls=":")
    ax.set_title(f"{title}  n={split_results[name]['n_used']}")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_xlim(tmin, tmax)
axes[0, 0].set_ylabel(f"Angular speed ({VELOCITY_UNIT})")
axes[1, 0].set_ylabel(f"Angular speed ({VELOCITY_UNIT})")
axes[1, 0].set_xlabel(f"Time from {ALIGN_TO} (ms)")
axes[1, 1].set_xlabel(f"Time from {ALIGN_TO} (ms)")
fig.suptitle("Split 2c averages (no isolation)")
fig.tight_layout()
_save_fig(fig, figures_dir / "split_averages.pdf")

# Overlay no_prev vs current for the largest-n bin if present
fig, ax = plt.subplots(figsize=(5.4, 3.4), dpi=120)
if current["series"] and split_results["no_prev"]["series"]:
    s0 = max(current["series"], key=lambda s: s["n_events"])
    # match bin label
    s1 = next((s for s in split_results["no_prev"]["series"] if s["raw_label"] == s0["raw_label"]), None)
    ax.plot(s0["time_axis"], s0["vel_center"], color="0.4", lw=2.0, label=f"all  {s0['label']}")
    if s1 is not None:
        ax.plot(s1["time_axis"], s1["vel_center"], color="C3", lw=2.0, label=f"no_prev  {s1['label']}")
ax.axvline(-80.0, color="0.75", lw=0.6, ls=":")
ax.legend(frameon=False, fontsize=8)
ax.set_title("Largest bin: all vs no previous neighbor")
ax.set_xlabel(f"Time from {ALIGN_TO} (ms)")
ax.set_ylabel(f"Angular speed ({VELOCITY_UNIT})")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
_save_fig(fig, figures_dir / "overlay_all_vs_no_prev.pdf")


no_prev    used= 8359  bins=6
has_prev   used= 8544  bins=6
no_next    used= 8352  bins=6
has_next   used= 8551  bins=6
isolated   used= 4010  bins=5
first      used= 4339  bins=6
last       used= 4332  bins=6
<IPython.core.display.Image object>
<IPython.core.display.Image object>


## 7. Signed movement-axis velocity

Unsigned speed hides direction. Same-direction pre-step → another saccade in a
sequence. Opposite-direction → square-wave / corrective / oscillation candidate.


In [17]:
signed_all = average_2c(events, eye_cache, isolation="none", signed=True)
signed_no_prev = average_2c(
    events, eye_cache, isolation="none", row_mask=~events["has_prev"].to_numpy(), signed=True
)
plot_speed_inline(
    signed_all["series"],
    "Signed speed along event movement axis (all events)",
    figures_dir / "signed_axis_all.pdf",
    signed=True,
)
plot_speed_inline(
    signed_no_prev["series"],
    "Signed speed along event movement axis (no previous neighbor)",
    figures_dir / "signed_axis_no_prev.pdf",
    signed=True,
)


<IPython.core.display.Image object>
<IPython.core.display.Image object>


## 8. Example raw traces


In [19]:
rng = np.random.default_rng(0)
# Prefer a mid/high amp bin so the main saccade is obvious.
edges, labels = _amp_edges(events[AMP_COL].to_numpy(dtype=float))
pick_bin = labels[min(2, len(labels) - 1)] if labels else None
ev_b = events.copy()
ev_b["amp_bin"] = pd.cut(ev_b[AMP_COL], bins=edges, labels=labels, include_lowest=True, right=False)

def _sample_rows(mask, n):
    pool = ev_b.index[mask]
    if len(pool) == 0:
        return []
    take = min(n, len(pool))
    return list(rng.choice(pool.to_numpy(), size=take, replace=False))


n_each = max(1, N_EXAMPLE_TRACES // 2)
idx_prev = _sample_rows((ev_b["has_prev"]) & (ev_b["amp_bin"] == pick_bin), n_each)
idx_iso = _sample_rows((~ev_b["has_prev"]) & (ev_b["amp_bin"] == pick_bin), n_each)

fig, axes = plt.subplots(2, n_each, figsize=(3.2 * n_each, 5.2), dpi=120, sharex=True, sharey=True)
if n_each == 1:
    axes = np.array(axes).reshape(2, 1)
for row_i, (idxs, title) in enumerate(((idx_prev, "has_prev"), (idx_iso, "no_prev"))):
    for col, idx in enumerate(idxs):
        ax = axes[row_i, col]
        row = ev_b.loc[idx]
        packed = eye_cache.get(_group_key(row["animal"], row["block"], row["eye"]))
        if packed is None:
            ax.set_title("no trace")
            continue
        t0 = float(row["peak_ms"])
        m = (packed["t_ms"] >= t0 + tmin) & (packed["t_ms"] <= t0 + tmax)
        ax.plot(packed["t_ms"][m] - t0, packed["v_vel"][m], color="0.15", lw=1.0)
        ax.axvspan(row["saccade_on_ms"] - t0, row["saccade_off_ms"] - t0, color="C0", alpha=0.15)
        if np.isfinite(row["prev_peak_lag_ms"]):
            ax.axvline(-float(row["prev_peak_lag_ms"]), color="C1", lw=0.8, ls="--")
        ax.axvline(0.0, color="0.5", lw=0.5, ls=":")
        ax.set_title(f"{title} {row['amp_bin']}\nlag_prev={row['prev_peak_lag_ms']:.0f} ms", fontsize=8)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.set_xlim(tmin, tmax)
axes[1, 0].set_xlabel(f"Time from {ALIGN_TO} (ms)")
axes[0, 0].set_ylabel(f"speed ({VELOCITY_UNIT})")
axes[1, 0].set_ylabel(f"speed ({VELOCITY_UNIT})")
fig.suptitle(f"Example traces, bin {pick_bin} (dashed = previous peak)")
fig.tight_layout()
_save_fig(fig, figures_dir / "example_raw_traces.pdf")


<IPython.core.display.Image object>


## 9. Isolation experiment + candidate PDF

`drop_events` keeps only isolated saccades (loses n). `nan_mask_neighbors` keeps
every event but occupancy-averages around neighboring on/off (NaN pad).
Candidate `figure_2c_isolated.pdf` uses `ISOLATION_MODE`.


In [21]:
iso_drop = average_2c(events, eye_cache, isolation="drop_events")
iso_nan = average_2c(events, eye_cache, isolation="nan_mask_neighbors")
print(f"drop_events         used={iso_drop['n_used']} dropped={iso_drop['n_dropped']} bins={len(iso_drop['series'])}")
print(f"nan_mask_neighbors  used={iso_nan['n_used']} dropped={iso_nan['n_dropped']} bins={len(iso_nan['series'])}")

modes = {
    "none": current,
    "drop_events": iso_drop,
    "nan_mask_neighbors": iso_nan,
}
candidate = modes[ISOLATION_MODE]

bump_all = pd.concat(
    [
        bump_metrics(current["series"]).assign(isolation="none"),
        bump_metrics(iso_drop["series"]).assign(isolation="drop_events"),
        bump_metrics(iso_nan["series"]).assign(isolation="nan_mask_neighbors"),
    ],
    ignore_index=True,
)
bump_all.to_csv(metadata_dir / "bump_metrics.csv", index=False)
print(bump_all.to_string(index=False))

plot_speed_panel(current["series"], animal, figures_dir / "figure_2c.pdf")
plot_speed_panel(candidate["series"], f"{animal} isolated ({ISOLATION_MODE})", figures_dir / "figure_2c_isolated.pdf")
plot_speed_inline(iso_drop["series"], "drop_events", figures_dir / "figure_2c_drop_events_inline.pdf")
plot_speed_inline(iso_nan["series"], "nan_mask_neighbors", figures_dir / "figure_2c_nan_mask_inline.pdf")

fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.2), dpi=120, sharex=True, sharey=True)
for ax, (name, res) in zip(axes, modes.items()):
    for s in res["series"]:
        ax.plot(s["time_axis"], s["vel_center"], color=s["color_rgba"], lw=1.5)
    ax.axvline(0.0, color="0.6", lw=0.6, ls="--")
    ax.axvline(-80.0, color="0.75", lw=0.6, ls=":")
    ax.set_title(f"{name}\nn={res['n_used']}")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_xlim(tmin, tmax)
axes[0].set_ylabel(f"Angular speed ({VELOCITY_UNIT})")
axes[1].set_xlabel(f"Time from {ALIGN_TO} (ms)")
fig.suptitle("Isolation comparison")
fig.tight_layout()
_save_fig(fig, figures_dir / "isolation_comparison.pdf")


drop_events         used=4260 dropped=12643 bins=6
nan_mask_neighbors  used=16899 dropped=4 bins=6
   bin  n_events  n_dropped  t_pre_ms      y_pre  t_post_ms     y_post     y_peak  pre_ratio  post_ratio          isolation
  0-5°     10497          0     -40.0 143.952148       40.0 132.175247 245.180893   0.587126    0.539093               none
 5-10°      3573          0     -52.0 170.125565       40.0 153.325928 376.963135   0.451306    0.406740               none
10-15°      1650          0     -56.0 174.519470       40.0 181.942337 453.127502   0.385144    0.401526               none
15-20°       793          0     -58.0 191.048325       40.0 208.851334 524.209656   0.364450    0.398412               none
20-25°       292          0     -54.0 216.932098       40.0 226.412109 571.844116   0.379355    0.395933               none
25-30°        98          0     -66.0 260.987366       40.0 276.318420 622.965820   0.418943    0.443553               none
  0-5°      2931       7566     -

## 10. Extra knobs (not expected to plant a −80 ms satellite)

Smoothing can broaden the main peak; it should not create a bump at −80 ms.
`ALIGN_TO='onset'` checks whether peak alignment is shifting a shoulder.


In [23]:
# Bandwidth sweep on the *current* (no isolation) kernel, by temporarily
# redefining sigma via a local average with BANDWIDTH override is awkward;
# re-smooth occupancy already stored on current series instead (cheap).
fig, ax = plt.subplots(figsize=(5.4, 3.4), dpi=120)
s0 = max(current["series"], key=lambda s: s["n_events"]) if current["series"] else None
if s0 is not None:
    occ = np.asarray(s0["occupancy"], dtype=float)
    # Reconstruct unsmoothed mean is not stored; show existing curve vs
    # lighter/heavier Gaussian on the already-smoothed mean as a visual check.
    y = np.asarray(s0["vel_center"], dtype=float)
    ax.plot(t_grid, y, color="0.2", lw=2.0, label=f"bandwidth={BANDWIDTH_MS} (used)")
    if BANDWIDTH_MS != 4:
        y4 = gaussian_filter1d(np.nan_to_num(y), sigma=max(1.0, 4.0 / DT_MS), mode="nearest")
        ax.plot(t_grid, y4, color="C0", lw=1.2, label="extra smooth σ≈4 ms on mean")
    ax.axvline(-80.0, color="0.75", lw=0.6, ls=":")
    ax.legend(frameon=False, fontsize=8)
ax.set_title("Smoothing cannot invent the −80 ms satellite")
ax.set_xlabel(f"Time from {ALIGN_TO} (ms)")
ax.set_ylabel(f"Angular speed ({VELOCITY_UNIT})")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
_save_fig(fig, figures_dir / "bandwidth_note.pdf")

print("To try ALIGN_TO='onset' or BANDWIDTH_MS=0/4, edit PARAMS and re-run from cell 1.")
print("FORCE_REDETECT is", FORCE_REDETECT)


<IPython.core.display.Image object>
To try ALIGN_TO='onset' or BANDWIDTH_MS=0/4, edit PARAMS and re-run from cell 1.
FORCE_REDETECT is False


## 11. Write LOGIC.md + numeric verdict


In [25]:
def _mean_pre_ratio(df, isolation):
    sub = df[df["isolation"] == isolation]
    if sub.empty:
        return np.nan
    return float(np.nanmean(sub["pre_ratio"]))


pre_none = _mean_pre_ratio(bump_all, "none")
pre_drop = _mean_pre_ratio(bump_all, "drop_events")
pre_nan = _mean_pre_ratio(bump_all, "nan_mask_neighbors")
frac_prev = float(events["has_prev"].mean())
frac_next = float(events["has_next"].mean())
n_iso = int((events["burst_role"] == "isolated").sum())

# Compare bump in has_prev vs no_prev for the shared bins
bp = bump_metrics(split_results["has_prev"]["series"])
bn = bump_metrics(split_results["no_prev"]["series"])
pre_has = float(np.nanmean(bp["pre_ratio"])) if len(bp) else np.nan
pre_no = float(np.nanmean(bn["pre_ratio"])) if len(bn) else np.nan

if np.isfinite(pre_has) and np.isfinite(pre_no) and pre_has > 1.5 * max(pre_no, 1e-6) and pre_no < 0.25:
    verdict = (
        "ARTIFACT: the −80 ms bump tracks previous-peak neighbors. Isolation "
        "(especially nan_mask_neighbors / drop_events) removes it. There is no "
        "matching post-peak satellite in isolated events, so this is not a 12.5 Hz "
        "saccadic rhythm."
    )
elif np.isfinite(pre_nan) and pre_nan > 0.25 and np.isfinite(pre_no) and pre_no > 0.25:
    verdict = (
        "UNRESOLVED / possible biology: a pre-peak bump remains after removing "
        "events with a previous neighbor. Do not claim a rhythm unless a matching "
        "post bump and opposite-direction signed pre-motion are also present."
    )
else:
    verdict = (
        "LIKELY ARTIFACT (mixed): neighbor-split and/or isolation reduce the pre-peak "
        "ratio. Treat remaining one-sided bumps as incomplete isolation, not a rhythm."
    )

print("frac has_prev/has_next:", round(frac_prev, 3), round(frac_next, 3))
print("mean pre_ratio none/drop/nan:", pre_none, pre_drop, pre_nan)
print("mean pre_ratio has_prev/no_prev:", pre_has, pre_no)
print("isolated events:", n_iso, "/", len(events))
print("VERDICT:", verdict)

lines = [
    "# Mouse 2c pre-peak bump",
    "",
    "## Setup",
    f"- Animal: M_002 blocks 012-015",
    f"- Align: {ALIGN_TO}, window {T_WINDOW_MS}, dt {DT_MS} ms, bandwidth {BANDWIDTH_MS} ms",
    f"- Isolation candidate: {ISOLATION_MODE}",
    f"- Events: {len(events)}  isolated: {n_iso}",
    "",
    "## Neighbor census",
    f"- Fraction with a previous same-eye peak in +/-{NEIGHBOR_RADIUS_MS:.0f} ms: {frac_prev:.3f}",
    f"- Fraction with a next same-eye peak in +/-{NEIGHBOR_RADIUS_MS:.0f} ms: {frac_next:.3f}",
    "",
    "## Pre-peak / main-peak height (mean across amp bins)",
    f"- all events: {pre_none:.3f}",
    f"- has_prev split: {pre_has:.3f}",
    f"- no_prev split: {pre_no:.3f}",
    f"- drop_events: {pre_drop:.3f}",
    f"- nan_mask_neighbors: {pre_nan:.3f}",
    "",
    "## Verdict",
    verdict,
    "",
    "## Files",
    "- plots/figure_2c.pdf — current averaging (paper-sized)",
    f"- plots/figure_2c_isolated.pdf — candidate using {ISOLATION_MODE}",
    "- metadata/event_neighbors.csv, bump_metrics.csv, neighbor_fractions_by_amp_bin.csv",
    "",
    "Production export_pos_vel_bundle was not modified.",
]
logic = "\n".join(lines) + "\n"
(OUT / "LOGIC.md").write_text(logic, encoding="utf-8")
(metadata_dir / "LOGIC.md").write_text(logic, encoding="utf-8")
print("wrote", OUT / "LOGIC.md")


frac has_prev/has_next: 0.488 0.488
mean pre_ratio none/drop/nan: 0.4310541660344136 0.2832252600268988 0.252629441626746
mean pre_ratio has_prev/no_prev: 0.7406197742193128 0.21233859838793176
isolated events: 4659 / 17585
VERDICT: ARTIFACT: the −80 ms bump tracks previous-peak neighbors. Isolation (especially nan_mask_neighbors / drop_events) removes it. There is no matching post-peak satellite in isolated events, so this is not a 12.5 Hz saccadic rhythm.
wrote /Users/nimi/Projects/PETS/outputs/review_answers_latest/mouse_2c_prepeak/LOGIC.md


## 12. Conclusion (M_002 run)

**Verdict: averaging artifact, not a biological ~12.5 Hz saccadic rhythm.**

Current 2c puts every finite sample in ±100 ms of peak speed into the mean. On M_002 (17,585 events, blocks 012–015):

- **48.8%** of events have another same-eye peak in the previous 100 ms (same fraction have one in the next 100 ms). That is leftover neighbors in the averaging window, not a rare subsequence.
- Median previous **onset ISI is 50 ms** (only 28% of previous ISIs fall in 60–100 ms). The mean’s pre-search maximum sits around **−50 to −66 ms** for larger amp bins (the 0–5° “−40 ms” hit is the search-window edge / main-peak shoulder). The eye-catching “−80 ms” wiggle on the paper plot is this neighbor mix after 10 ms Gaussian smoothing, not a sharp 80 ms clock.
- Split test: mean pre/peak height is **0.74 with a previous neighbor vs 0.21 without**. Isolation drops the same ratio from **0.43 (all)** to **0.28 (drop_events)** / **0.25 (nan_mask_neighbors)**. Isolated traces do not grow a matching post-peak satellite.
- Signed velocity along the movement axis does not show a reversing oscillation before t = 0 once previous neighbors are removed.

**Candidate plot:** `outputs/review_answers_latest/mouse_2c_prepeak/plots/figure_2c_isolated.pdf` using `ISOLATION_MODE = "nan_mask_neighbors"` (keeps n; occupancy-averages around other events’ on/off). `drop_events` is cleaner but throws away most events (~4,260 vs ~16,900).

**Not done (on purpose):** production `export_pos_vel_bundle` is unchanged. Single animal. Detection threshold was not re-swept (`FORCE_REDETECT = False`).

Re-fidget from the PARAMS cell: `NEIGHBOR_RADIUS_MS`, `NEIGHBOR_PAD_MS`, `ISOLATION_MODE`, `BANDWIDTH_MS`, `ALIGN_TO`. See `LOGIC.md` in the plot folder for the numbers from this run.

**Decision rule (for later animals / knobs)**

| Observation | Interpretation |
|---|---|
| Bump in `has_prev`, gone in `no_prev`; isolation removes it; no matching post bump when isolated | Averaging artifact from previous saccades in the ±100 ms window |
| Bump remains when isolated **and** a similar post bump appears **and** signed pre-motion reverses | Possible biological oscillation; needs more animals |
| Bump remains when isolated but is one-sided / same-direction | Incomplete isolation or detection merging — try `NEIGHBOR_PAD_MS`, not a rhythm claim |
